# Dissertation Code Walkthrough: Reuben Data Height Growth Models

This notebook compiles the work in this directory into a readable workflow: understanding the data, showing the key exploratory plots, explaining the model implementations, summarising the result tables, and interpreting the residual/terrain/remote-sensing analysis.

The main result is **Table 4.1**, the temporal common-plot experiment: train on 2012 plot measurements and predict the corresponding 2023 plot heights. The other tables are included because they check different generalisation settings.


## 1. Project Structure and Workflow

The repository is organised around a small shared `common/` package, separate model folders under `models/`, saved artefacts under `outputs/`, and post-hoc analysis scripts under `data_exploration/`.

High-level workflow:

1. Load and purge the data, splitting rows into 2012 and 2023.
2. Train/evaluate four model families: `AvgByAge`, `Chapman-Richards`, `DNN`, and `PINN`.
3. Save comparable metrics using the same definitions: MAE, MSE, R², MRE, and accuracy.
4. Generate spatial error maps and model-comparison plots.
5. Analyse residuals against terrain and remote-sensing variables to understand where errors remain.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_PURGED = ROOT / "data/Aber_1223_conjoin_plot_purged_expanded_encoded.csv"
DATA_UNSEEN = ROOT / "data/Aber_1223_conjoin_expanded_encoded_duplicate_plots_removed.csv"
OUTPUTS = ROOT / "outputs"

pd.set_option("display.max_columns", 80)
pd.set_option("display.precision", 4)


## 2. Understanding the Data

There are two CSVs in `data/`:

- **Purged common-plot CSV**: used for Table 4.1, 4.3, and 4.4. It contains the same 27,660 plots in 2012 and 2023 after removing negative-growth/noisy common plots.
- **Unseen duplicate-removed CSV**: used for Table 4.2. It contains extra 2012-only and 2023-only plots, so Table 4.2 is a larger temporal generalisation test rather than the same paired common-plot setup.

Saved data summary:

| Dataset | Year | Rows | Unique plots | Height min | Height mean | Height max | Age min | Age mean | Age max |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Purged common-plot CSV | 2012 | 27660 | 27660 | 5.4745 | 21.3215 | 49.7867 | 20.0000 | 37.1950 | 83.0000 |
| Purged common-plot CSV | 2023 | 27660 | 27660 | 9.2418 | 27.8716 | 51.9592 | 20.0000 | 48.1687 | 94.0000 |
| Unseen duplicate-removed CSV | 2012 | 32351 | 32351 | 5.4745 | 21.5367 | 49.7867 | 20.0000 | 38.1467 | 83.0000 |
| Unseen duplicate-removed CSV | 2023 | 42696 | 42696 | 5.6357 | 25.2378 | 51.9592 | 20.0000 | 43.6800 | 113.0000 |

The 2023 heights are higher on average in the paired purged data, which matches the growth-prediction setup: the models learn from the 2012 state and are evaluated against later top height.


In [ ]:
df_purged = pd.read_csv(DATA_PURGED)
df_unseen = pd.read_csv(DATA_UNSEEN)

summary = (
    df_purged.groupby("YEAR")[["TOP_HEIGHT", "AGE"]]
    .agg(["count", "mean", "std", "min", "max"])
)
summary


### Feature Set

The shared feature list is deliberately compact and reproducible. `PLOT_ID` is only used to match plots across years and is not used as a model feature.

| Type | Columns |
| --- | --- |
| Numeric | `X`, `Y`, `AGE` |
| Cultivation dummy columns | `CULTIVATN_MOUNDING`, `CULTIVATN_NO CULTIVATION` |
| Primary land-use dummy columns | 13 `PRILANDUSE_*` columns |
| Target | `TOP_HEIGHT` |

The age-only baselines (`AvgByAge` and `Chapman-Richards`) use only `AGE`; the neural models use all 18 shared features.


### What the Data Looks Like

The following plots are generated from the purged common-plot CSV and saved in `notebook_assets/` so they display without rerunning the notebook.


![TOP_HEIGHT distribution](notebook_supplementary_figures/notebook_assets__height_distribution.svg)

Interpretation: the 2023 distribution is shifted upward relative to 2012, consistent with forest growth over the temporal interval. The overlap also shows why the task is not trivial: a plot's age and local characteristics still matter.


![Age-height scatter](notebook_supplementary_figures/notebook_assets__age_height_scatter.svg)

Interpretation: height generally increases with age but with large scatter. This is why a pure age curve can be useful but incomplete: plots of the same age can still have very different heights.


![Spatial layout](notebook_supplementary_figures/notebook_assets__spatial_layout.svg)

Interpretation: plot locations are spatially structured rather than uniformly random. This motivates the spatial error maps and later residual analysis against terrain and satellite-derived features.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for year, colour in [(2012, "tab:blue"), (2023, "tab:orange")]:
    subset = df_purged[df_purged["YEAR"] == year]
    axes[0].hist(subset["TOP_HEIGHT"], bins=30, alpha=0.55, label=str(year), color=colour)
    axes[1].scatter(subset["AGE"], subset["TOP_HEIGHT"], s=4, alpha=0.15, label=str(year), color=colour)

axes[0].set_title("Top-height distribution")
axes[0].set_xlabel("TOP_HEIGHT (m)")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].set_title("Age vs top height")
axes[1].set_xlabel("AGE")
axes[1].set_ylabel("TOP_HEIGHT (m)")

sample = df_purged.sample(min(8000, len(df_purged)), random_state=42)
axes[2].scatter(sample["X"], sample["Y"], c=sample["YEAR"], s=3, alpha=0.35, cmap="coolwarm")
axes[2].set_title("Sampled spatial layout")
axes[2].set_xlabel("X")
axes[2].set_ylabel("Y")
plt.tight_layout()


## 3. Experimental Tables

The result tables use the same metric definitions across models:

- **MAE**: mean absolute error in metres.
- **MSE**: mean squared error.
- **R²**: coefficient of determination; higher is better, and negative values mean the model is worse than predicting the test-set mean.
- **MRE**: mean relative error, `mean(abs(actual - predicted) / actual)`.
- **Acc%**: `(1 - MRE) * 100`.

Table meanings:

- **Table 4.1**: main temporal common-plot experiment; 2012 paired plots train, 2023 paired plots test.
- **Table 4.2**: unseen/full temporal experiment; all post-purge 2012 rows train, all post-purge 2023 rows test, including 13,832 2023-only plots.
- **Table 4.3**: 3-fold cross-validation within 2012.
- **Table 4.4**: 3-fold cross-validation within 2023.


In [ ]:
def load_results():
    model_map = {
        "avg_by_age": "AvgByAge",
        "chapman_richards": "CR",
        "dnn_baseline": "DNN",
        "pinn_baseline": "PINN",
    }
    metric_map = {"mae": "MAE", "mse": "MSE", "r2": "R²", "mre": "MRE", "acc": "Acc%"}
    records = []
    for folder, model in model_map.items():
        df = pd.read_csv(OUTPUTS / folder / "results.csv")
        for _, row in df.iterrows():
            table, metric = row["Metric"].split("_")
            records.append({
                "Table": table.replace("Table", ""),
                "Model": model,
                "Metric": metric_map[metric],
                "Value": row["Value"],
            })
    return pd.DataFrame(records)

results = load_results()
results.pivot_table(index=["Table", "Model"], columns="Metric", values="Value")


## 4. Main Results: Table 4.1

Table 4.1 is the main result because every model is evaluated on the same paired 2023 plots after training on the corresponding 2012 population.

| Model | MAE | MSE | R² | MRE | Acc% |
| --- | --- | --- | --- | --- | --- |
| AvgByAge | 5.8100 | 48.9849 | -0.1400 | 0.2170 | 78.3028 |
| CR | 4.9872 | 35.1121 | 0.1829 | 0.1915 | 80.8472 |
| DNN | 5.3450 | 43.9719 | -0.0233 | 0.2087 | 79.1310 |
| PINN | 4.5184 | 30.2234 | 0.2966 | 0.1758 | 82.4180 |

Interpretation:

- **PINN is the strongest local replication** in Table 4.1: MAE = 4.5184 m, MSE = 30.2234, R² = 0.2966, and accuracy = 82.4180%.
- **Chapman-Richards is the strongest simple baseline** among the age-only methods, improving substantially over `AvgByAge`.
- **DNN underperforms PINN here**, despite using the full 18-feature set. This suggests the physics-informed growth-rate constraint is useful in the temporal extrapolation setting.
- `AvgByAge` has negative R², meaning age-bucket means alone are not enough for reliable 2023 prediction.


### Comparison with Reuben's Original Table 4.1

Local replication values are close in some baselines but lower for the neural models than Reuben's original reported values.

| Model | MAE | MSE | R² | MRE | Acc% |
| --- | --- | --- | --- | --- | --- |
| AvgByAge | 5.8121 | 48.9792 | -0.1398 | 0.2170 | 78.2900 |
| LinReg | 4.9311 | 35.4522 | 0.1750 | 0.1911 | 80.8900 |
| CR | 4.5592 | 31.0253 | 0.2780 | 0.1844 | 81.5600 |
| RF | 5.2237 | 41.0254 | 0.0453 | 0.1959 | 80.4100 |
| DNN | 4.4835 | 30.5735 | 0.2885 | 0.1764 | 82.3600 |
| PINN | 4.0424 | 24.1570 | 0.4378 | 0.1539 | 84.6100 |

The audit notes in `audit.md` argue that the comparison should be described as a comparison between **model pipelines**, not solely architectures: age-only empirical/parametric baselines differ from full-feature neural models in both feature set and model family.


### Table 4.1 Spatial Error Plots

The combined spatial plots show whether model errors cluster geographically. Unsigned relative error shows error size; signed relative error shows under-prediction vs over-prediction.


![Combined spatial relative error](notebook_supplementary_figures/data_exploration__model_comparison_output__combined_spatial_error.png)

Interpretation: spatial clustering remains visible across models, so the residual error is not purely random measurement noise. The PINN reduces overall error but does not remove all geographic structure.


![Combined spatial signed error](notebook_supplementary_figures/data_exploration__model_comparison_output__combined_spatial_signed_error.png)

Interpretation: signed maps help distinguish areas where models systematically under-predict from areas where they over-predict. That distinction is important for diagnosing missing predictors such as terrain, exposure, soil, or canopy condition.


![Model difference plot](notebook_supplementary_figures/data_exploration__model_comparison_output__compare_plot_subtract.png)

Interpretation: this difference plot is useful for seeing where one model's error surface improves on another, rather than just comparing aggregate metrics.


## 5. Other Result Tables

### Table 4.2: Unseen/full temporal test

| Model | MAE | MSE | R² | MRE | Acc% |
| --- | --- | --- | --- | --- | --- |
| AvgByAge | 5.3170 | 42.6851 | 0.2565 | 0.2214 | 77.8617 |
| CR | 4.7628 | 33.2597 | 0.4206 | 0.2026 | 79.7369 |
| DNN | 5.3310 | 48.0439 | 0.1631 | 0.2470 | 75.3046 |
| PINN | 4.3637 | 28.2432 | 0.5080 | 0.1896 | 81.0439 |

Interpretation: Table 4.2 is harder and broader than Table 4.1 because the 2023 test set contains many plots not present in 2012. PINN remains best overall here, with the highest R² and accuracy. Chapman-Richards remains a strong simple baseline. DNN performs poorly on this broader temporal transfer in the saved run, especially on MRE/accuracy.

### Table 4.3: 3-fold CV within 2012

| Model | MAE | MSE | R² | MRE | Acc% |
| --- | --- | --- | --- | --- | --- |
| AvgByAge | 3.3535 | 17.9776 | 0.6054 | 0.1854 | 81.4636 |
| CR | 3.7216 | 21.6505 | 0.5248 | 0.2052 | 79.4796 |
| DNN | 2.9885 | 14.7948 | 0.6752 | 0.1610 | 83.9047 |
| PINN | 2.7042 | 12.5268 | 0.7250 | 0.1446 | 85.5382 |

Interpretation: this is an interpolation-style check within the 2012 distribution. Neural models do better here than the age-only baselines because feature-rich models can exploit within-year structure. PINN is best in the saved local results, with DNN close behind.

### Table 4.4: 3-fold CV within 2023

| Model | MAE | MSE | R² | MRE | Acc% |
| --- | --- | --- | --- | --- | --- |
| AvgByAge | 3.9888 | 25.4897 | 0.4068 | 0.1670 | 83.2957 |
| CR | 4.3551 | 29.7407 | 0.3079 | 0.1830 | 81.6988 |
| DNN | 3.3832 | 23.4880 | 0.4540 | 0.1382 | 86.1844 |
| PINN | 3.2285 | 17.8066 | 0.5855 | 0.1313 | 86.8658 |

Interpretation: this is the equivalent within-year check for 2023. PINN again has the best saved local metrics, while DNN is second. The gap between within-year CV and temporal transfer reinforces that predicting across time is harder than interpolating within a year.


## 6. Model-by-Model Implementation Notes

### AvgByAge

`models/avg_by_age/train.py` implements the simplest baseline:

1. Load 2012 and 2023 rows.
2. Group 2012 rows by `AGE`.
3. Compute mean 2012 `TOP_HEIGHT` for each age.
4. For every 2023 row, use the exact age if available or snap to the nearest valid 2012 age.

This is interpretable but coarse. It assumes an average tree of the same age in 2012 is a reasonable proxy for a 2023 tree, ignoring spatial, land-use, and stand-condition differences.


### Chapman-Richards

`models/chapman_richards/train.py` fits a parametric growth curve:

```text
H(t) = y_max * (1 - exp(-k * t)) ** p
```

The fitted Table 4.1 parameters saved in `outputs/chapman_richards/cr_params.json` are:

```json
{"y_max": 46.60095257234912, "k": 0.01687239415474734, "p": 0.9917917417072227}
```

This model is still age-only, but unlike `AvgByAge` it encodes a smooth biological growth shape. It also provides the physics prior used by the PINN.


![Chapman-Richards fitted curve](notebook_supplementary_figures/outputs__chapman_richards__cr_fit.png)

Interpretation: the curve captures the broad age-height relationship but cannot explain plot-level deviations caused by local conditions.


### DNN Baseline

`models/dnn_baseline/model.py` defines a feed-forward regression network:

```text
18 input features -> 128 -> 128 -> 128 -> 1 output
```

Training details saved in `outputs/dnn_baseline/config_used.json`:

- Optimiser: Adam.
- Learning rate: 0.0001.
- Batch size: 512.
- Max epochs: 1000.
- Validation split: 0.33 of the 2012 training rows.
- Regularisation: small L1 penalty.
- Scheduler: ReduceLROnPlateau.

The DNN is flexible and uses all shared features, but it has no explicit growth-rate constraint.


![DNN training curve](notebook_supplementary_figures/outputs__dnn_baseline__training_curve.png)

Interpretation: the training curve is the diagnostic for convergence and overfitting. The model is evaluated only after predictions are inverse-transformed back to metres.


### PINN Baseline

`models/pinn_baseline/model.py` uses the same broad neural architecture as the DNN, but keeps `AGE` as a separate tensor so PyTorch autograd can compute `d(predicted height) / d(AGE)`.

The loss is:

```text
loss = data MSE + lambda_ph * physics MSE + lambda_l1 * L1 penalty
```

where the physics term compares the network's learned age derivative with the analytical derivative of the Chapman-Richards curve. In the saved configuration, `lambda_ph = 1.0` and `lambda_l1 = 1e-5`.

This explains why the PINN can outperform the DNN in temporal prediction: it is still feature-rich, but its learned growth behaviour is regularised toward a plausible biological curve.


![PINN training curve](notebook_supplementary_figures/outputs__pinn_baseline__training_curve.png)

Interpretation: the PINN training curve combines the usual data-fitting behaviour with a physics-informed constraint. The final Table 4.1 saved result is stronger than the plain DNN on all reported metrics.


## 7. Residual and Terrain Analysis

After the main models were trained, the project analysed where residuals remain. The important point is that this is exploratory interpretation, not a new final prediction model.

Terrain variables were added from DEM data:

- `DEM_ELEVATION`
- `DEM_RUGGEDNESS`

The residual tables compare these variables against signed residuals and relative errors for CR and PINN.


### Terrain Correlations

| Terrain feature | Target | n | Pearson r | Spearman r |
| --- | --- | --- | --- | --- |
| DEM_ELEVATION | CR_RESIDUAL_2023 | 27660 | -0.3990 | -0.3739 |
| DEM_ELEVATION | PINN_RESIDUAL_2023 | 27660 | -0.2509 | -0.2091 |
| DEM_ELEVATION | PINN_REL_ERROR | 27660 | 0.0367 | -0.0286 |
| DEM_ELEVATION | CR_REL_ERROR | 27660 | 0.0793 | -0.1048 |
| DEM_ELEVATION | X | 27660 | -0.4615 | -0.5082 |
| DEM_ELEVATION | Y | 27660 | 0.2884 | 0.2906 |
| DEM_RUGGEDNESS | CR_RESIDUAL_2023 | 27660 | 0.1587 | 0.1432 |
| DEM_RUGGEDNESS | PINN_RESIDUAL_2023 | 27660 | 0.1464 | 0.1401 |

Interpretation: elevation has a clear negative association with CR residuals and a weaker but still visible association with PINN residuals. In practical terms, the age-only growth curve misses terrain-related structure more strongly than the PINN, but the PINN still leaves some terrain signal unexplained.


### Terrain Bins

| Group type | Group | n | CR residual mean | PINN residual mean | PINN rel err mean | CR rel err mean |
| --- | --- | --- | --- | --- | --- | --- |
| ELEVATION_BIN | 0-100m | 5611 | 4.1355 | 2.5819 | 0.1634 | 0.2160 |
| ELEVATION_BIN | 100-200m | 12285 | 3.2136 | 3.0313 | 0.1670 | 0.1773 |
| ELEVATION_BIN | 200-300m | 5189 | 2.2971 | 2.4517 | 0.1436 | 0.1486 |
| ELEVATION_BIN | 300-400m | 3541 | -1.7230 | 0.4010 | 0.1453 | 0.2140 |
| ELEVATION_BIN | 400m+ | 1034 | -5.4603 | -4.1917 | 0.2935 | 0.3666 |
| RUGGEDNESS_QUARTILE | Q1 lowest | 6915 | 0.8357 | 0.9973 | 0.1890 | 0.2290 |
| RUGGEDNESS_QUARTILE | Q2 | 6915 | 2.2157 | 2.2599 | 0.1621 | 0.1947 |
| RUGGEDNESS_QUARTILE | Q3 | 6915 | 2.7439 | 2.6975 | 0.1594 | 0.1804 |
| RUGGEDNESS_QUARTILE | Q4 highest | 6915 | 3.2945 | 2.9439 | 0.1448 | 0.1621 |

Interpretation: the high-elevation bin (`400m+`) has much larger relative errors for both CR and PINN. This suggests that high-elevation stands may behave differently from the overall growth pattern captured by the core features.


![Terrain maps](notebook_supplementary_figures/data_exploration__terrain_exploration_output__terrain_maps.png)

![Terrain residual scatter](notebook_supplementary_figures/data_exploration__terrain_exploration_output__terrain_residual_scatter.png)

![Terrain binned errors](notebook_supplementary_figures/data_exploration__terrain_exploration_output__terrain_binned_errors.png)


## 8. Residual Feature Importance

The residual feature-importance scripts quantify whether missing variables can explain the remaining model errors. Ridge/linear feature-set checks are used here as diagnostics.

| Target | Feature set | n features | CV R² | CV MAE | CV RMSE |
| --- | --- | --- | --- | --- | --- |
| CR_RESIDUAL_2023 | DEM only | 2 | 0.2132 | 3.7647 | 4.8543 |
| CR_RESIDUAL_2023 | Existing PINN features | 18 | 0.0519 | 4.2176 | 5.3285 |
| CR_RESIDUAL_2023 | All features + DEM | 20 | 0.2384 | 3.6969 | 4.7759 |
| PINN_RESIDUAL_2023 | DEM only | 2 | 0.1003 | 3.5135 | 4.5087 |
| PINN_RESIDUAL_2023 | Existing PINN features | 18 | 0.0321 | 3.6485 | 4.6766 |
| PINN_RESIDUAL_2023 | All features + DEM | 20 | 0.1333 | 3.4282 | 4.4253 |

Interpretation: DEM-only features explain more of the CR residual variation than the existing PINN features alone. For PINN residuals the signal is smaller, but DEM still explains non-trivial variation. This supports the idea that terrain is a plausible missing covariate.


![CR residual feature-set R2](notebook_supplementary_figures/data_exploration__residual_feature_importance_output__linear_feature_set_cv_r2_cr_residual_2023.png)

![PINN residual feature-set R2](notebook_supplementary_figures/data_exploration__residual_feature_importance_output__linear_feature_set_cv_r2_pinn_residual_2023.png)


## 9. GEE / Remote-Sensing Exploration

The AlphaEarth/GEE exploration adds Sentinel-2 bands/indices and Sentinel-1 SAR features, then checks whether these explain residuals beyond DEM.

| Target | DEM R² | GEE R² | GEE+DEM R² | Gain over DEM |
| --- | --- | --- | --- | --- |
| CR_RESIDUAL_2023 | 0.2132 | 0.3188 | 0.4178 | 0.2046 |
| PINN_RESIDUAL_2023 | 0.1003 | 0.2509 | 0.2954 | 0.1951 |

Interpretation: GEE features add residual signal beyond DEM for both CR and PINN residuals. The combined `GEE + DEM` feature set performs best in these exploratory linear checks, especially for CR residuals. This suggests canopy/spectral condition and terrain together explain part of what the original model inputs miss.


![GEE vs DEM summary](notebook_supplementary_figures/data_exploration__alpha_earth_exploration_output__gee_vs_dem_bar.png)

![GEE spatial maps](notebook_supplementary_figures/data_exploration__alpha_earth_exploration_output__gee_spatial_maps.png)

![Top GEE correlations for CR residuals](notebook_supplementary_figures/data_exploration__alpha_earth_exploration_output__top_gee_correlations_cr_residual_2023.png)

![Top GEE correlations for PINN residuals](notebook_supplementary_figures/data_exploration__alpha_earth_exploration_output__top_gee_correlations_pinn_residual_2023.png)


## 10. Rough Implementation Summary

The implementation is intentionally modular:

1. `common/config.py` defines shared paths, feature names, target column, and random seed.
2. `common/data_utils.py` loads data, removes negative-growth common plots, builds feature arrays, splits validation folds, and prepares scalers/tensors.
3. `common/metrics.py` centralises metric definitions so every model is evaluated identically.
4. Each model folder contains its own `train.py`, `model.py` where relevant, `config.py`, and plotting helpers.
5. Outputs are saved per model under `outputs/<model_name>/`, making comparison scripts straightforward.
6. Post-hoc scripts under `data_exploration/` read saved outputs and generate comparison, residual, terrain, and GEE figures.

The most important methodological caveat is that the four models are not a pure architecture-only ablation. The baselines use age-only modelling, while DNN/PINN use a larger shared feature set. The cleaner framing is: **the project compares intentionally different modelling pipelines under common table-specific evaluation protocols.**


## 11. Main Takeaways

- Table 4.1 is the headline common-plot temporal experiment.
- PINN is the strongest saved local model across the main table and the saved CV tables.
- Chapman-Richards is the strongest simple age-only baseline and supplies the PINN physics prior.
- DNN benefits from feature richness in within-year CV but is weaker than PINN in temporal generalisation.
- Residual maps and terrain/GEE analysis show that errors are spatially structured and partly explainable by missing environmental/remote-sensing information.
- Future work could add DEM and remote-sensing variables directly to the predictive model, then test whether those gains persist under the same Table 4.1/4.2 protocols.
